#Installing Libraries

In [1]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install networkx
%pip install "huggingface_hub==0.25.2"
!pip install pytesseract
!pip install requests

!git clone https://github.com/boomb0om/CRAFT-text-detection.git
!git clone https://github.com/madmaze/pytesseract.git

import os
import sys
sys.path.append('/content/CRAFT-text-detection')

import yaml
import cv2
import torch
import numpy as np
import networkx as nx
import pytesseract
import re
import requests
import huggingface_hub
import gradio as gr

from ultralytics import YOLO
from matplotlib import pyplot as plt
from collections import defaultdict
from CRAFT import CRAFTModel, draw_polygons
from PIL import Image
from shapely.geometry import Point, Polygon, LineString
from huggingface_hub import get_token, whoami

print("-----All libraries installed-----")

fatal: destination path 'CRAFT-text-detection' already exists and is not an empty directory.
fatal: destination path 'pytesseract' already exists and is not an empty directory.
-----All libraries installed-----


#YOLO Symbol Detection

In [3]:
# Unzip images to a custom data folder
!unzip -q /content/Kia_Sportage_Labeled_Images.zip -d /content/custom_data
!mv /content/custom_data/Kia_Sportage_Labeled_Images/images /content/custom_data
!mv /content/custom_data/Kia_Sportage_Labeled_Images/labels /content/custom_data
!mv /content/custom_data/Kia_Sportage_Labeled_Images/classes.txt /content/custom_data

!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.8

# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

# !yolo detect train data=/content/data.yaml model=yolo11m.pt epochs=60 imgsz=640

mv: cannot stat '/content/custom_data/Kia_Sportage_Labeled_Images/images': No such file or directory
mv: cannot stat '/content/custom_data/Kia_Sportage_Labeled_Images/labels': No such file or directory
mv: cannot stat '/content/custom_data/Kia_Sportage_Labeled_Images/classes.txt': No such file or directory
--2026-10-25 21:45:25--  https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3203 (3.1K) [text/plain]
Saving to: ‘/content/train_val_split.py’

/content/train_val_ 100%[===================>]   3.13K  --.-KB/s    in 0s      

2026-10-25 21:45:25 (39.7 MB/s) - ‘/content/train_val_split.py’ saved [3203/3203]

Created folder at /content/data/trai

# Line Detection

In [6]:

# Install the missing extraction utility
!apt-get update -qq
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Verify installation
!ollama --version


import os
import time
import subprocess
import requests

OLLAMA_BASE_URL = os.environ.get(
    "OLLAMA_BASE_URL", "http://127.0.0.1:11434"
)
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3.2")
OLLAMA_TIMEOUT_SECONDS = 120
OLLAMA_LAST_STATUS = "Not run yet"

# Start Ollama in the background
server_log = open("/tmp/ollama.log", "w")
server_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# Wait for the API to respond
ollama_ready = False

for attempt in range(30):
    try:
        response = requests.get(
            f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags",
            timeout=3
        )
        if response.ok:
            ollama_ready = True
            break
    except requests.RequestException:
        pass

    if server_process.poll() is not None:
        break

    time.sleep(1)

if not ollama_ready:
    server_log.close()
    print("Ollama failed to start. Recent log:")
    !tail -n 40 /tmp/ollama.log
    raise RuntimeError("Ollama server did not become available.")

# Download the model
subprocess.run(
    ["ollama", "pull", OLLAMA_MODEL],
    check=True
)

# Verify the model is available
models_response = requests.get(
    f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags",
    timeout=10
)
models_response.raise_for_status()

available_models = [
    model.get("name", "")
    for model in models_response.json().get("models", [])
]

if not any(
    name == OLLAMA_MODEL or name.startswith(OLLAMA_MODEL + ":")
    for name in available_models
):
    raise RuntimeError(
        f"Model {OLLAMA_MODEL!r} is not available. "
        f"Installed models: {available_models}"
    )

print(f"Ollama is running at {OLLAMA_BASE_URL}")
print(f"Configured model: {OLLAMA_MODEL}")
print(f"Available models: {available_models}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 15 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama is running at http://127.0.0.1:11434
Configured model: llama3.2
Available models: ['llama3.2:latest']


In [19]:
import os
import re
import time
from collections import Counter, defaultdict

import cv2
import gradio as gr
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pytesseract
import requests
from PIL import Image
from shapely.geometry import LineString, Point, Polygon

# Expected to be imported/instantiated in your environment:
# from ultralytics import YOLO
# from <your_craft_module> import CRAFTModel

# ============================================================
# 1. MODEL AND CONFIGURATION
# ============================================================

model = YOLO("/content/best.pt")

with open("/content/custom_data/classes.txt", "r", encoding="utf-8") as f:
    class_names = [line.strip() for line in f if line.strip()]

ignored_classes = {""}

# Tolerance between wire endpoints and symbol bounding boxes
margin = 25
threshold = 50
minLineLength = 0
maxLineGap = 20

# Visualization options
showSymbolLabel = True
showLineLabel = False

# Ollama Defaults
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3"
OLLAMA_TIMEOUT_SECONDS = 30
OLLAMA_LAST_STATUS = "NOT RUN"

ANNOTATED_PATH = "/content/annotated_output.png"
GRAPH_PATH = "/content/connection_graph.png"
SUMMARY_PATH = "/content/schematic_summary.txt"
CONNECTIONS_PATH = os.path.abspath("output.txt")


# ============================================================
# 2. GEOMETRY HELPERS
# ============================================================

def unpack_line(line):
    """Safely unpacks coordinates from any shape (flat tuple, list, or nested 2D array)."""
    coords = np.asarray(line, dtype=np.int32).flatten()
    if coords.size != 4:
        raise ValueError(
            f"Expected 4 coordinates; got shape {np.asarray(line).shape}, value={line!r}"
        )
    return int(coords[0]), int(coords[1]), int(coords[2]), int(coords[3])


def is_point_near_box(point, box, margin=30):
    px, py = point
    x1, y1, x2, y2 = unpack_line(box)
    return (
        x1 - margin <= px <= x2 + margin
        and y1 - margin <= py <= y2 + margin
    )


def ccw(a, b, c):
    return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])


def do_lines_intersect(a1, a2, b1, b2):
    return (
        ccw(a1, b1, b2) != ccw(a2, b1, b2)
        and ccw(a1, a2, b1) != ccw(a1, a2, b2)
    )


def does_line_intersect_box(p1, p2, box):
    x1, y1, x2, y2 = unpack_line(box)
    rect_lines = [
        ((x1, y1), (x2, y1)),
        ((x2, y1), (x2, y2)),
        ((x2, y2), (x1, y2)),
        ((x1, y2), (x1, y1)),
    ]
    return any(do_lines_intersect(p1, p2, rp1, rp2) for rp1, rp2 in rect_lines)


# ============================================================
# 3. OCR AND SYMBOL LABELING
# ============================================================

craft_model = CRAFTModel(
    "weights/",
    "cpu",
    use_refiner=False,
    fp16=False
)


def recognize_text_from_polygons(image, polygons):
    texts = []
    image_h, image_w = image.shape[:2]

    for poly in polygons:
        pts = np.array(poly, dtype=np.int32)
        x, y, w, h = cv2.boundingRect(pts)

        x1 = max(0, x)
        y1 = max(0, y)
        x2 = min(image_w, x + w)
        y2 = min(image_h, y + h)
        roi = image[y1:y2, x1:x2]

        if roi.size == 0:
            continue

        text = pytesseract.image_to_string(
            roi,
            config="--psm 7"
        ).strip()

        if text:
            texts.append((poly, text))

    return texts


def find_merged_text(text, match_case, symbol_box, recognized_text):
    match = re.match(match_case, text, re.IGNORECASE)
    if not match:
        return None

    text_below = match.group()
    x1_s, y1_s, x2_s, y2_s = unpack_line(symbol_box)
    text_above = ""
    min_distance = float("inf")

    for poly, recognized in recognized_text:
        px, py, pw, ph = cv2.boundingRect(np.array(poly, dtype=np.int32))
        x1_t, y1_t, x2_t, y2_t = px, py, px + pw, py + ph

        vertical_overlap = (y2_t >= y1_s - 15 and y2_t <= y1_s)
        horizontal_overlap = (x1_t <= x2_s and x2_t >= x1_s)

        if vertical_overlap and horizontal_overlap:
            distance = y1_s - y2_t
            if distance < min_distance:
                min_distance = distance
                text_above = recognized.strip().lower()

    return f"{text_above}_{text_below}" if text_above else text_below


def validate_symbol_name(ocr_text, class_name, box, recognized_text):
    text = ocr_text.replace(" ", "")
    lower_class = class_name.lower()

    if lower_class == "relay":
        return find_merged_text(text, r"relay", box, recognized_text)

    if lower_class == "ground":
        match = re.search(r"G[O0]\d", ocr_text, re.IGNORECASE)
        return match.group() if match else None

    if lower_class == "motor":
        return ocr_text.capitalize() if "motor" in text.lower() else None

    if "fuse" in lower_class:
        return find_merged_text(text, r"^\d{2,3}A$", box, recognized_text)

    if "connector" in lower_class:
        return None

    return None


def assign_symbol_ids_with_text(
    symbol_boxes,
    recognized_text,
    symbol_classes,
    class_names,
    margin=25
):
    symbol_ids = []
    label_count = defaultdict(int)

    for idx, box in enumerate(symbol_boxes):
        x1, y1, x2, y2 = unpack_line(box)
        class_name = class_names[symbol_classes[idx]]
        assigned_text = None

        for poly, text in recognized_text:
            px, py, pw, ph = cv2.boundingRect(np.array(poly, dtype=np.int32))
            text_cx = px + pw / 2
            text_cy = py + ph / 2

            if (
                x1 - margin <= text_cx <= x2 + margin
                and y1 - margin <= text_cy <= y2 + margin
            ):
                valid_text = validate_symbol_name(
                    text,
                    class_name,
                    box,
                    recognized_text
                )
                if valid_text:
                    assigned_text = valid_text
                    break

        if not assigned_text:
            label_count[class_name] += 1
            symbol_ids.append(f"{class_name}_{label_count[class_name]}")
        else:
            symbol_ids.append(assigned_text)

    for idx, label in enumerate(symbol_ids):
        class_name = class_names[symbol_classes[idx]].lower()
        if "connector" not in class_name:
            print(label)

    return symbol_ids


# ============================================================
# 4. IMAGE AND WIRE HELPERS
# ============================================================

def remove_lines_overlapping_polygons(lines, polygons):
    def line_intersects_polygon(p1, p2, poly):
        poly_shape = Polygon(poly)
        return (
            poly_shape.contains(Point(p1))
            or poly_shape.contains(Point(p2))
            or poly_shape.intersects(LineString([p1, p2]))
        )

    kept_lines = []
    for line in lines:
        x1, y1, x2, y2 = unpack_line(line)
        if not any(
            line_intersects_polygon((x1, y1), (x2, y2), poly)
            for poly in polygons
        ):
            kept_lines.append(line)

    return kept_lines


def draw_label(image, text, x, y, color=(255, 200, 0), scale=0.65):
    font = cv2.FONT_HERSHEY_SIMPLEX
    thickness = 2

    (tw, th), baseline = cv2.getTextSize(str(text), font, scale, thickness)

    y = max(y, th + baseline + 8)
    x = max(x, 2)
    image_h, image_w = image.shape[:2]
    x = min(x, max(2, image_w - tw - 12))

    cv2.rectangle(
        image,
        (x, y - th - baseline - 8),
        (min(image_w - 1, x + tw + 10), min(image_h - 1, y + 2)),
        (30, 35, 45),
        -1
    )

    cv2.putText(
        image,
        str(text),
        (x + 5, y - 4),
        font,
        scale,
        color,
        thickness,
        cv2.LINE_AA
    )


def detect_lines(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blur, 50, 150, apertureSize=3)

    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=threshold,
        minLineLength=minLineLength,
        maxLineGap=maxLineGap
    )

    return lines if lines is not None else []


def build_line_graph(lines):
    graph = nx.Graph()
    for line in lines:
        x1, y1, x2, y2 = unpack_line(line)
        graph.add_edge((x1, y1), (x2, y2))
    return graph


# ============================================================
# 5. SCHEMATIC SUMMARY
# ============================================================

def build_schematic_summary(symbol_ids, symbol_classes, class_names, connections):
    class_counts = Counter(class_names[class_id] for class_id in symbol_classes)

    graph = nx.Graph()
    graph.add_nodes_from(symbol_ids)
    graph.add_edges_from(connections)

    total_symbols = len(symbol_ids)
    total_connections = len(connections)
    connected_symbols = sum(1 for node in graph.nodes if graph.degree(node) > 0)
    isolated_symbols = sorted(node for node in graph.nodes if graph.degree(node) == 0)
    groups = sorted(
        (group for group in nx.connected_components(graph) if len(group) > 1),
        key=len,
        reverse=True
    )

    lines = [
        "SCHEMATIC SUMMARY",
        "=" * 44,
        f"Detected components: {total_symbols}",
        f"Inferred component-to-component connections: {total_connections}",
        f"Components with inferred connections: {connected_symbols}",
        f"Components with no inferred connections: {len(isolated_symbols)}",
        f"Connected groups: {len(groups)}",
        "",
        "COMPONENT INVENTORY",
        "-" * 44,
    ]

    if class_counts:
        for name, count in class_counts.most_common():
            lines.append(f"• {name}: {count}")
    else:
        lines.append("No symbols were detected.")

    lines.extend(["", "CONNECTIVITY INSIGHTS", "-" * 44])

    if groups:
        for idx, group in enumerate(groups[:8], start=1):
            members = sorted(group)
            member_text = ", ".join(members[:12])
            if len(members) > 12:
                member_text += f" ... (+{len(members) - 12} more)"
            group_edges = graph.subgraph(group).number_of_edges()
            lines.append(
                f"Group {idx}: {len(members)} components, "
                f"{group_edges} connections"
            )
            lines.append(f"  {member_text}")
    else:
        lines.append("No multi-component connections were inferred.")

    if isolated_symbols:
        lines.extend(["", "COMPONENTS WITH NO INFERRED CONNECTIONS"])
        isolated_text = ", ".join(isolated_symbols[:20])
        if len(isolated_symbols) > 20:
            isolated_text += f" ... (+{len(isolated_symbols) - 20} more)"
        lines.append(isolated_text)

    lines.extend([
        "",
        "INTERPRETATION",
        "-" * 44,
        f"The vision pipeline detected {total_symbols} components and inferred {total_connections} component-to-component connections.",
        "Components with no inferred connections may be genuinely unconnected, or their wires may have been missed during detection.",
        "This is a computer-vision summary, not a verification of electrical correctness or circuit function."
    ])

    return "\n".join(lines)


def summarize_with_ollama(factual_summary):
    global OLLAMA_LAST_STATUS

    prompt = f"""You are an electrical-diagram explainer for someone with no technical background.

Your job is to explain what this diagram appears to do, not to repeat a list of components and connections.

Write a clear explanation with these sections:
1. What this diagram appears to do
2. How the main parts work together
3. What we can and cannot tell from the image

Rules:
- Start with the big picture and explain the likely purpose of the circuit in everyday language, if the evidence supports it.
- Explain how important components contribute to the system. For example, a relay acts like an electrically controlled switch, a fuse helps protect a circuit, and a motor creates movement.
- Explain what the connections mean in context. Do not merely list component counts, positions, coordinates, or graph groups.
- Do not invent what the circuit powers, what activates it, or its overall purpose when the evidence is insufficient.
- Clearly distinguish connections inferred by the software from verified electrical connections.
- Do not claim specific current flow, voltage, or circuit behavior unless the input establishes it.
- Briefly explain any important uncertainty caused by missed components, labels, or wires.
- Use everyday language and define unavoidable technical terms.
- Focus on explaining the system, not counting its parts.
- Keep the response concise, around 120–200 words.
- Return only the explanation.

SOURCE FACTS:
{factual_summary}
"""

    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL.rstrip('/')}/api/generate",
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.2}
            },
            timeout=OLLAMA_TIMEOUT_SECONDS
        )
        response.raise_for_status()
        result = response.json().get("response", "").strip()
        if not result:
            OLLAMA_LAST_STATUS = "RULE-BASED FALLBACK (Ollama returned an empty response)"
            return factual_summary

        OLLAMA_LAST_STATUS = f"OLLAMA GENERATED ({OLLAMA_MODEL})"
        return result
    except Exception as exc:
        OLLAMA_LAST_STATUS = f"RULE-BASED FALLBACK (Ollama unavailable: {type(exc).__name__})"
        return factual_summary


# ============================================================
# 6. MAIN DETECTION PIPELINE
# ============================================================

def detect_connections(input_path, draw_debug_points=True, show_plot=True):
    with open(CONNECTIONS_PATH, "w", encoding="utf-8") as out_file:
        if (
            os.path.isfile(input_path)
            and input_path.lower().endswith((".jpg", ".jpeg", ".png"))
        ):
            image_files = [input_path]
        elif os.path.isdir(input_path):
            image_files = [
                os.path.join(input_path, filename)
                for filename in sorted(os.listdir(input_path))
                if filename.lower().endswith((".jpg", ".jpeg", ".png"))
            ]
        else:
            raise ValueError("Invalid input path. Provide an image file or a folder.")

        if not image_files:
            raise ValueError("No supported images were found.")

        for img_path in image_files:
            filename = os.path.basename(img_path)
            out_file.write(f"Image: {filename}\n")
            image = cv2.imread(img_path)

            if image is None:
                print(f"Could not read image: {img_path}")
                continue

            # --- 0. Detect text boxes (CRAFT) ---
            pil_img = Image.open(img_path).convert("RGB")
            polygons = craft_model.get_polygons(pil_img)
            recognized_text = recognize_text_from_polygons(image, polygons)

            # --- 1. Symbol detection (YOLO) ---
            results = model(img_path)
            symbol_boxes, symbol_classes = [], []

            for result in results:
                for box in result.boxes:
                    coords = box.xyxy[0].cpu().numpy().astype(int)
                    cls_id = int(box.cls[0])
                    if class_names[cls_id] in ignored_classes:
                        continue
                    symbol_boxes.append(tuple(coords))
                    symbol_classes.append(cls_id)

            symbol_ids = assign_symbol_ids_with_text(
                symbol_boxes, recognized_text, symbol_classes,
                class_names, margin=50
            )

            # --- 2. Detect and filter lines ---
            wire_lines = remove_lines_overlapping_polygons(
                detect_lines(image), polygons
            )

            # --- 3. Keep wires with an endpoint near a symbol ---
            filtered_lines = []
            for line in wire_lines:
                x1, y1, x2, y2 = unpack_line(line)
                p1, p2 = (x1, y1), (x2, y2)
                length = ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5

                if length < minLineLength:
                    continue

                if any(
                    is_point_near_box(point, box, margin)
                    for box in symbol_boxes
                    for point in (p1, p2)
                ):
                    filtered_lines.append(line)

            wire_lines = filtered_lines
            line_graph = build_line_graph(wire_lines)

            # --- 4. Map wire endpoints to nearby symbols ---
            endpoint_to_symbols = {}
            for point in line_graph.nodes:
                for idx, box in enumerate(symbol_boxes):
                    if is_point_near_box(point, box, margin=50):
                        endpoint_to_symbols.setdefault(point, set()).add(idx)

            # --- 5. Build symbol connectivity ---
            connections = set()
            for wire_cluster in nx.connected_components(line_graph):
                connected_symbols = set()
                for point in wire_cluster:
                    connected_symbols.update(endpoint_to_symbols.get(point, []))

                if len(connected_symbols) >= 2:
                    names = sorted({symbol_ids[idx] for idx in connected_symbols})
                    for i in range(len(names)):
                        for j in range(i + 1, len(names)):
                            connections.add(tuple(sorted((names[i], names[j]))))

            # --- 5.4. Generate the schematic summary ---
            summary_text = build_schematic_summary(
                symbol_ids, symbol_classes, class_names, connections
            )
            summary_text = summarize_with_ollama(summary_text)
            summary_text = f"SUMMARY ENGINE: {OLLAMA_LAST_STATUS}\n" + ("=" * 44) + "\n\n" + summary_text

            with open(SUMMARY_PATH, "w", encoding="utf-8") as summary_file:
                summary_file.write(summary_text)

            # --- 5.5. Create the component-level connectivity graph ---
            component_graph = nx.Graph()
            component_graph.add_nodes_from(symbol_ids)
            component_graph.add_edges_from(connections)

            plt.close("all")

            if component_graph.number_of_nodes() > 0:
                node_count = component_graph.number_of_nodes()
                edge_count = component_graph.number_of_edges()

                def compact_graph_label(name):
                    name = str(name)
                    name = re.sub(r"^Pigtail Connector", "PC", name)
                    name = re.sub(r"^Direct Connector", "DC", name)
                    name = re.sub(r"^Diagram Connector", "DiagC", name)
                    name = re.sub(r"^Ground", "GND", name)
                    return name.replace("_", "-")

                graph_labels = {
                    node: compact_graph_label(node)
                    for node in component_graph.nodes()
                }

                groups = sorted(
                    nx.connected_components(component_graph),
                    key=len,
                    reverse=True
                )

                group_count = len(groups)
                cols = min(3, max(1, group_count))
                rows = (group_count + cols - 1) // cols

                fig, axes = plt.subplots(
                    rows,
                    cols,
                    figsize=(7 * cols, 5.5 * rows),
                    squeeze=False
                )
                axes_flat = axes.flatten()

                for group_idx, group in enumerate(groups):
                    ax = axes_flat[group_idx]
                    subgraph = component_graph.subgraph(group).copy()
                    n = subgraph.number_of_nodes()

                    pos = nx.spring_layout(
                        subgraph,
                        seed=42,
                        k=1.8 / max(n ** 0.5, 1),
                        iterations=150
                    )

                    nx.draw_networkx_edges(
                        subgraph, pos, ax=ax,
                        edge_color="#64748b",
                        width=1.6,
                        alpha=0.7
                    )

                    nx.draw_networkx_nodes(
                        subgraph, pos, ax=ax,
                        node_size=max(350, min(1000, 18000 / max(n, 1))),
                        node_color="#67e8f9",
                        edgecolors="#0f172a",
                        linewidths=1.5
                    )

                    nx.draw_networkx_labels(
                        subgraph,
                        pos,
                        labels={node: graph_labels[node] for node in subgraph.nodes()},
                        ax=ax,
                        font_size=max(6, min(10, 12 - (n // 5))),
                        font_weight="medium",
                        font_color="#0f172a",
                        bbox={
                            "facecolor": "white",
                            "edgecolor": "none",
                            "alpha": 0.9,
                            "pad": 2
                        }
                    )

                    ax.set_title(
                        f"Group {group_idx + 1} | {n} components, {subgraph.number_of_edges()} connections",
                        fontsize=11,
                        fontweight="bold",
                        pad=12
                    )
                    ax.axis("off")

                for ax in axes_flat[group_count:]:
                    ax.axis("off")

                fig.suptitle(
                    f"Electrical Connectivity Map | {node_count} components | {edge_count} connections",
                    fontsize=16,
                    fontweight="bold",
                    y=1.01
                )
                fig.tight_layout()
            else:
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.text(0.5, 0.5, "No symbols detected", ha="center", va="center", fontsize=14)
                ax.set_title("Electrical Connectivity Map")
                ax.axis("off")

            plt.savefig(
                GRAPH_PATH,
                dpi=220,
                bbox_inches="tight",
                facecolor="white"
            )
            if show_plot:
                plt.show()
            plt.close("all")

            # --- 6. Annotate schematic ---
            for idx, box in enumerate(symbol_boxes):
                x1, y1, x2, y2 = unpack_line(box)
                label = symbol_ids[idx]
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                if showSymbolLabel:
                    draw_label(image, label, x1, y1 - 5)

            for idx, line in enumerate(wire_lines):
                x1, y1, x2, y2 = unpack_line(line)
                cv2.line(image, (x1, y1), (x2, y2), (0, 0, 255), 2)

                if showLineLabel:
                    cv2.putText(
                        image,
                        f"line{idx + 1}",
                        (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.75,
                        (59, 71, 110),
                        1
                    )

                if draw_debug_points:
                    for point in ((x1, y1), (x2, y2)):
                        cv2.circle(image, point, 3, (255, 0, 0), -1)

            for poly in polygons:
                pts = np.array(poly, np.int32).reshape((-1, 1, 2))
                cv2.polylines(
                    image,
                    [pts],
                    isClosed=True,
                    color=(255, 0, 255),
                    thickness=2
                )

            cv2.imwrite(ANNOTATED_PATH, image)

            # --- 7. Filter wires connected to at least one symbol ---
            connected_wires = [
                (pt1, pt2)
                for pt1, pt2 in line_graph.edges
                if (pt1 in endpoint_to_symbols or pt2 in endpoint_to_symbols)
            ]
            filtered_graph = nx.Graph()
            filtered_graph.add_edges_from(connected_wires)

            # --- 8. Assign wire IDs ---
            wire_ids = {
                (pt1, pt2): f"wire_{idx + 1}"
                for idx, (pt1, pt2) in enumerate(filtered_graph.edges)
            }
            wire_ids.update({
                (pt2, pt1): label
                for (pt1, pt2), label in list(wire_ids.items())
            })

            # --- 9. Map points to symbols ---
            point_to_symbol = {
                point: idx
                for point, indices in endpoint_to_symbols.items()
                for idx in indices
            }

            # --- 10. Trace connections into the text report ---
            printed = set()
            for component in nx.connected_components(filtered_graph):
                symbols_in_comp = [
                    (point, point_to_symbol[point])
                    for point in component
                    if point in point_to_symbol
                ]

                for i in range(len(symbols_in_comp)):
                    for j in range(i + 1, len(symbols_in_comp)):
                        pt1, idx1 = symbols_in_comp[i]
                        pt2, idx2 = symbols_in_comp[j]
                        label1, label2 = symbol_ids[idx1], symbol_ids[idx2]

                        if label1 == label2:
                            continue

                        try:
                            path = nx.shortest_path(filtered_graph, pt1, pt2)
                            steps = [label1] + [
                                wire_ids.get((path[k], path[k + 1]), "wire_?")
                                for k in range(len(path) - 1)
                            ] + [label2]

                            path_str = " → ".join(steps)
                            pair = tuple(sorted((label1, label2)))

                            if pair not in printed:
                                print(path_str)
                                out_file.write("  " + path_str + "\n")
                                printed.add(pair)

                        except nx.NetworkXNoPath:
                            continue


# ============================================================
# 7. GRADIO INTERFACE
# ============================================================

def analyze_schematic(image_path):
    if image_path is None:
        raise gr.Error("Please upload an electrical schematic image.")

    for path in (
        ANNOTATED_PATH,
        GRAPH_PATH,
        SUMMARY_PATH,
        CONNECTIONS_PATH
    ):
        if os.path.exists(path):
            os.remove(path)

    try:
        print("Running detection pipeline...")
        total_start = time.time()
        detect_connections(image_path, show_plot=False)
        print(f"Total processing time: {time.time() - total_start:.2f}s")

        expected_files = [
            (ANNOTATED_PATH, "Annotated schematic"),
            (GRAPH_PATH, "Connection graph"),
            (SUMMARY_PATH, "Schematic summary"),
            (CONNECTIONS_PATH, "Connection report")
        ]

        for path, description in expected_files:
            if not os.path.exists(path):
                raise gr.Error(f"{description} was not generated.")

        with Image.open(ANNOTATED_PATH) as img:
            annotated = img.convert("RGB")

        with Image.open(GRAPH_PATH) as img:
            graph_image = img.convert("RGB")

        with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
            summary_text = f.read()

        with open(CONNECTIONS_PATH, "r", encoding="utf-8") as f:
            connection_text = f.read()

        return (
            annotated,
            graph_image,
            summary_text,
            connection_text,
            CONNECTIONS_PATH
        )

    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Analysis failed: {e}")


custom_css = """
.gradio-container {
    max-width: 1450px !important;
    margin: auto !important;
    font-family: Inter, Arial, sans-serif !important;
}
footer {
    display: none !important;
}
h1 {
    letter-spacing: -1px;
}
label, .prose {
    line-height: 1.6;
}
.section-heading {
    background: #cffafe;
    color: #0891b2;
    padding: 8px 12px;
    border-radius: 8px;
    font-weight: 700;
    margin-bottom: 8px;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css
) as demo:
    gr.Markdown("# CircuitVision")
    gr.Markdown(
        "Electrical schematic intelligence | Symbol detection · OCR · "
        "Wire tracing · Connectivity mapping · Automatic summary"
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown(
                '<div class="section-heading">01 — Upload electrical schematic</div>'
            )
            image_input = gr.Image(
                type="filepath",
                label="Electrical schematic",
                show_label=False
            )
            with gr.Row():
                clear_btn = gr.ClearButton(
                    components=[image_input],
                    value="Clear"
                )
                analyze_btn = gr.Button(
                    "Analyze schematic",
                    variant="primary"
                )

        with gr.Column(scale=1):
            gr.Markdown(
                '<div class="section-heading">02 — Detected symbols, text &amp; wires</div>'
            )
            annotated_output = gr.Image(
                label="Annotated schematic",
                show_label=False
            )

    with gr.Row():
        with gr.Column():
            gr.Markdown(
                '<div class="section-heading">03 — Component connectivity graph</div>'
            )
            graph_output = gr.Image(
                label="Component connectivity graph",
                show_label=False
            )
        with gr.Column():
            gr.Markdown(
                '<div class="section-heading">04 — Schematic Summary</div>'
            )
            summary_output = gr.Textbox(
                label="Schematic summary",
                lines=14,
                interactive=False,
                show_label=False
            )

    with gr.Row():
        with gr.Column():
            gr.Markdown(
                '<div class="section-heading">05 — Inferred connections</div>'
            )
            connections_output = gr.Textbox(
                label="Inferred connections",
                lines=18,
                interactive=False,
                show_label=False
            )
        with gr.Column():
            gr.Markdown(
                '<div class="section-heading">06 — Download connection report</div>'
            )
            report_output = gr.File(
                label="06 — Download connection report",
                show_label=False
            )

    analyze_btn.click(
        fn=analyze_schematic,
        inputs=[image_input],
        outputs=[
            annotated_output,
            graph_output,
            summary_output,
            connections_output,
            report_output
        ]
    )

demo.queue()
demo.launch(share=True)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'cached_download' (from 'huggingface_hub.file_download') is deprecated and will be removed from version '0.26'. Use `hf_hub_download` instead.
  warnings.warn(warning_message, FutureWarning)
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:672: FutureWarning: 'cached_download' is the legacy way to download files from the HF hub, please consider upgrading to 'hf_hub_download'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=No

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc889d7f43b234e2cf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
